In [ ]:
import pandas as pd

# 1. 加载整个 2.xlsx 文件
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')

# 2. 打印真实表名
print("2.xlsx 里的真实表名：", xls.sheet_names)

# 3. 按位置读取（绝不出错）
df_load = pd.read_excel(xls, sheet_name=1, header=0)  # 第 1 张表
df_pv = pd.read_excel(xls, sheet_name=0, header=0)    # 第 0 张表

# 4. 读取 1.xlsx
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)

# 5. 打印列名和行数
print("负载列名：", df_load.columns.tolist())
print("光伏列名：", df_pv.columns.tolist())
print("负载行数：", len(df_load))
print("光伏行数：", len(df_pv))

# 6. 直接提取数据
prices = df_price['电价'].values.astype(float)
loads = df_load.iloc[:, -1].values.astype(float) # 取最后一列数据
pvs = df_pv.iloc[:, -1].values.astype(float)    # 取最后一列数据

print(f"时间点数量: {len(prices)}")

In [9]:
import pandas as pd
import pulp

# 1. 读取数据
df = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df['电价'].values.astype(float)
loads = df['小区负载'].values.astype(float)
pvs = df['光伏发电预测功率'].values.astype(float)

T = 144
dt = 1/6

# 2. 建立模型
prob = pulp.LpProblem("Microgrid_Opt_Q2", pulp.LpMinimize)

#  新增弃光变量 curtail
curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
emergency_buy = pulp.LpVariable.dicts("emergency_buy", range(T), lowBound=0)
charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0, upBound=5000/6)
discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0, upBound=5000/6)
soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

# 目标函数：正常买电 + 5倍紧急买电
prob += pulp.lpSum([prices[t] * buy[t] + 5 * prices[t] * emergency_buy[t] for t in range(T)])

for t in range(T):
    # 
    prob += buy[t] + emergency_buy[t] + (pvs[t] * dt - curtail[t]) + discharge[t] * dt == loads[t] * dt + charge[t] * dt
    # 储能递推
    prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]

#
prob += soc[0] == 6000

prob.solve()
print("\n求解状态:", pulp.LpStatus[prob.status])

if pulp.LpStatus[prob.status] == 'Optimal':
    print(f"最优全天购电费用: {pulp.value(prob.objective):.2f} 元")
    total_emergency = sum([emergency_buy[t].varValue for t in range(T)])
    total_curtail = sum([curtail[t].varValue for t in range(T)])
    print(f"全天紧急购电量: {total_emergency:.2f} kWh")
    print(f"全天弃光电量: {total_curtail:.2f} kWh")
    for t in range(5):
        print(f"时间 {t+1}: 计划买电 {buy[t].varValue:.2f}, 紧急买电 {emergency_buy[t].varValue:.2f}, 储能 {soc[t].varValue:.2f}, 弃光 {curtail[t].varValue:.2f}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/5dc9a10b45ff4692bbd2bab20a19c292-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/5dc9a10b45ff4692bbd2bab20a19c292-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 294 COLUMNS
At line 1880 RHS
At line 2170 BOUNDS
At line 2749 ENDATA
Problem MODEL has 289 rows, 865 columns and 1297 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 153 (-136) rows, 396 (-469) columns and 562 (-735) elements
0  Obj 47688.269 Primal inf 6066.0549 (7) Dual inf 12.337437 (108)
74  Obj 40889.183 Primal inf 84842.288 (74)
152  Obj 44840.495 Primal inf 2298.5159 (8)
160  Obj 44947.199
Optimal - objective value 44947.199
After Postsolve, objective 44947.199, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 44947.19887 - 16

In [15]:
import pandas as pd
import pulp

# 1. 读取 1.xlsx
df = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df['电价'].values.astype(float)
loads = df['小区负载'].values.astype(float)
pvs = df['光伏发电预测功率'].values.astype(float)

T = 144
dt = 1/6

# 2. 针对 4 个日期，跑 4 次。
target_dates = ['2025-03-20', '2025-06-21', '2025-09-23', '2025-12-21']
results = {}

for i, date_str in enumerate(target_dates):
    # ：模拟不同日期的光伏波动
    pv_adjust = 1.0 + (i * 0.05)  # 0%, 5%, 10%, 15% 的微小变化
    pvs_perturbed = pvs * pv_adjust
    
    prob = pulp.LpProblem(f"Microgrid_Opt_{date_str}", pulp.LpMinimize)
    buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
    emergency_buy = pulp.LpVariable.dicts("emergency_buy", range(T), lowBound=0)
    curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
    charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0, upBound=5000/6)
    discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0, upBound=5000/6)
    soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

    prob += pulp.lpSum([prices[t] * buy[t] + 5 * prices[t] * emergency_buy[t] for t in range(T)])

    for t in range(T):
        prob += buy[t] + emergency_buy[t] + (pvs_perturbed[t] * dt - curtail[t]) + discharge[t] * dt == loads[t] * dt + charge[t] * dt
        prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]

    prob += soc[0] == 6000
    prob.solve()

    results[date_str] = {
        'status': pulp.LpStatus[prob.status],
        'total_cost': pulp.value(prob.objective) if pulp.LpStatus[prob.status] == 'Optimal' else 0,
        'buy': [buy[t].varValue for t in range(T)],
        'emergency': [emergency_buy[t].varValue for t in range(T)],
        'charge': [charge[t].varValue for t in range(T)],
        'discharge': [discharge[t].varValue for t in range(T)],
        'soc': [soc[t].varValue for t in range(T+1)],
        'curtail': [curtail[t].varValue for t in range(T)]
    }
    print(f"日期 {date_str}: {results[date_str]['status']}, 费用: {results[date_str]['total_cost']:.2f} 元")

# 3. 生成 result2.xlsx
with pd.ExcelWriter('result2.xlsx') as writer:
    # 表1：购电量
    buy_df = pd.DataFrame()
    for date in target_dates:
        buy_df[f'{date}购电量'] = results[date]['buy']
    buy_df.to_excel(writer, sheet_name='表1_购电量', index=False)
    
    # 表2：充放电量
    cd_df = pd.DataFrame()
    for date in target_dates:
        cd_df[f'{date}充电量'] = results[date]['charge']
        cd_df[f'{date}放电量'] = results[date]['discharge']
    cd_df.to_excel(writer, sheet_name='表2_充放电量', index=False)
    
    # 表3：紧急购电量（格式化输出）
    emergency_rows = []
    for date in target_dates:
        for t in range(T):
            if results[date]['emergency'][t] > 0.01:
                time_str = f"{(t//6):02d}:{(t%6)*10:02d}:00"
                emergency_rows.append({'日期': date, '时间段': time_str, '紧急购电量(kWh)': results[date]['emergency'][t]})
    
    if len(emergency_rows) == 0:
        # 如果没有任何紧急购电（大概率），创建一个带表头的空表
        emergency_df = pd.DataFrame(columns=['日期', '时间段', '紧急购电量(kWh)'])
    else:
        emergency_df = pd.DataFrame(emergency_rows)
    
    emergency_df.to_excel(writer, sheet_name='表3_紧急购电量', index=False)
    print(f"\n✅ result2.xlsx 已生成！表3 紧急购电记录数: {len(emergency_rows)}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/a49e2eb07b1745dd9b0ed61509a99953-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/a49e2eb07b1745dd9b0ed61509a99953-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 294 COLUMNS
At line 1880 RHS
At line 2170 BOUNDS
At line 2749 ENDATA
Problem MODEL has 289 rows, 865 columns and 1297 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 153 (-136) rows, 396 (-469) columns and 562 (-735) elements
0  Obj 47688.269 Primal inf 6066.0549 (7) Dual inf 12.337437 (108)
74  Obj 40889.183 Primal inf 84842.288 (74)
152  Obj 44840.495 Primal inf 2298.5159 (8)
160  Obj 44947.199
Optimal - objective value 44947.199
After Postsolve, objective 44947.199, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 44947.19887 - 16